In [ ]:
import sys

sys.path.append("/home/ubuntu/work/therapeutic_accelerator/custom_packages/utils")
sys.path.append("/home/ubuntu/work/therapeutic_accelerator/custom_packages/database")

In [ ]:
# Base packages -----------------------------------------------------------------------------
import pandas as pd
import numpy as np
import logging
import re
import ast
from tqdm import tqdm
from tqdm.notebook import tqdm_notebook

# Paraellel Processing -----------------------------------------------------------------------
import dask
from dask import distributed, dataframe as dd
from dask.diagnostics import ProgressBar

# Model work --------------------------------------------------------------------------------
from transformers import AutoModel, AutoTokenizer
import torch


In [ ]:
# hide warnings

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Disable TensorFlow INFO and WARNING messages
import tensorflow as tf
tf.get_logger().setLevel('ERROR')  # Disable TensorFlow ERROR messages (optional)

import warnings
# Filter out Dask client warnings
warnings.filterwarnings("ignore")


In [ ]:
import chromaDB as cbd # specter_ef, create_text_splitter, create_chroma_client
from db_tools import db_connection
from utils import import_config

config, keys = import_config()

# Chroma setup --------------------------------------------------------------------------------
chroma_server_host = "3.86.216.30"
chroma_client = cbd.create_chroma_client(chroma_server_host)

# Modelish stuff --------------------------------------------------------------------------------

model = AutoModel.from_pretrained("allenai/specter")
tokenizer = AutoTokenizer.from_pretrained("allenai/specter")

# Create embeddings function with specter model
specter_embeder = cbd.specter_ef(model, tokenizer)

specter_embeder.create_text_splitter()


In [ ]:
# Working Collection
collection = chroma_client.get_or_create_collection("specter_abstracts", embedding_function=specter_embeder)

collection.count()

In [ ]:
def remove_exisiting(df):
    """ Remove existing ids from dataframe """
    # get unique ids from collection that are the corpus ids
    existing_ids = np.unique(pd.DataFrame(collection.get(include = ['metadatas']))['ids'].str.split('-',expand=True).rename(columns={0:'id',1:'part'})['id'].tolist()).tolist()
    print('N existing ids: ',len(existing_ids))
    print(df.shape)
    
    df = df[~df['corpusId'].astype(str).isin(existing_ids)]
    
    print(df.shape)
    
    return df

In [ ]:
abstracts = pd.read_csv("/home/ubuntu/work/data/abstracts.csv")
abstracts = remove_exisiting(abstracts)

Test

Functionalized

In [ ]:
# tester = abstracts.to_dict('records')
# len(tester)

In [ ]:
def create_ids_metadatas(corpusid, num_chunks): 
    """ Create ids for each chunk """
    
    ids = [f"{corpusid}-{i}" for i in range(num_chunks)]
    
    metadatas = [{'corpusid': corpusid, 'chunk':i} for i in range(num_chunks)]
    
    return ids, metadatas

In [ ]:
def process_dict(d):
    
    corpusid = d['corpusId']
    
    documents = specter_embeder.split_text(d['documents'])
    
    ids, metadatas = create_ids_metadatas(corpusid, len(documents))
    
    embeddings = specter_embeder.embed_documents(documents)

    new_dict = {
        'ids': ids,
        'documents': documents, 
        'metadatas': metadatas,
        'embeddings': embeddings
    }

    try: 
        collection.upsert(**new_dict)
        return True
    except: 
        return False


# Dask functions

In [ ]:
# Create dask cluster
dask.config.set(scheduler='processes')  # overwrite default with multiprocessing scheduler

cluster = distributed.LocalCluster(name='local', n_workers=5, memory_limit = '5GiB', threads_per_worker=1)  # Launches a scheduler and workers locally

client = distributed.Client(cluster)

client

# Add documents to collection

- map ddf partitions
- apply function over 
- convert row to dictionary
- process with `process_dict`

In [ ]:
# client.restart()

In [ ]:
# ab_tester = abstracts.iloc[:10000, :]
n_partitions = len(abstracts) // 100
print(n_partitions)

abstracts_dict = abstracts.to_dict('records')

chunked_abstracts = np.array_split(abstracts_dict, n_partitions)

In [ ]:
partition_size = 10

for i in tqdm_notebook(range(len(chunked_abstracts)), desc='chunk'): 
    
    dict_chunk = chunked_abstracts[i]
    
    # Register the ProgressBar with Dask
    # pbar.register()
    
    # dict_chunk = client.scatter(dict_chunk, broadcast=True)

    futures = [dask.delayed(process_dict)(d) for d in dict_chunk]

    num_partitions = len(futures)//partition_size
    
    futures_split = np.array_split(futures, num_partitions)
    
    for j in tqdm_notebook(range(len(futures_split)), leave = False, desc='partition'):
        
        fs = futures_split[j]
        
        results = dask.compute(*fs, traverse=False)
        
        del results
        
    # After your Dask computation is done, remove the ProgressBar
    # pbar.unregister()
        
    del dict_chunk, futures, futures_split

2023-07-27 10:31:56,392 - distributed.nanny - WARNING - Restarting worker
